# PriceCheck — Dataset Exploration
**ICS 3202: Artificial Intelligence — Project Deliverable 1**

**Group problem area:** A mobile app that lets shoppers in **Nairobi** check whether the price of an item at a supermarket/market is fair, by comparing it against a model trained on historical price data, powered by an ML price-prediction engine.

**Dataset explored:** Kenya - Food Prices, from the World Food Programme (WFP) Price Database, distributed via the Humanitarian Data Exchange (HDX) — filtered down to **Nairobi** markets only.

## 1. Candidate open-source datasets

| # | Dataset | Description | Expected output variable |
|---|---------|-------------|---------------------------|
| 1 | **Kenya - Food Prices** (WFP / HDX, [data.humdata.org/dataset/wfp-food-prices-for-kenya](https://data.humdata.org/dataset/wfp-food-prices-for-kenya)) | Monthly price records for staple commodities (maize, rice, beans, sugar, etc.) across ~380 markets in Kenya, from 2006 to present. Directly downloadable CSV, no login required. | `price` (or `usdprice`) — the recorded price of a commodity at a given market and date |
| 2 | **Kenya Supermarkets Data** (Kaggle, Emmanuel Kens) | 2017 point-of-sale-style transaction records from Kenyan supermarkets: item, category, unit price, quantity | `unit_price` — the price of a specific item |
| 3 | **Kenya: Average Retail Market Prices of Selected Food Crops** (KNBS, via HDX) | Average retail prices of food crops (maize, beans, cabbages, tomatoes, etc.) 2014-2018 | `average_retail_price` |
| 4 | **Kenya - Consumer Price Indices (CPI) and Inflation Rates** (KNBS, via HDX) | Monthly CPI and inflation rate by commodity category | `cpi` / `inflation_rate` |

### Why the WFP Kenya Food Prices dataset was selected
- It is the largest and most current of the candidates: monthly data from **2006 to the present**, updated every month.
- Its `market` field includes **Nairobi**, so it can be filtered down to just Nairobi price observations, matching our group's scope.
- It has a clean, directly downloadable CSV with no authentication required, unlike the Kaggle alternative which needs API credentials.
- `price` is a clear, continuous output variable, which fits a price-prediction/price-check ML engine well.

**Note:** no open dataset scraped directly from branded Nairobi supermarkets (Naivas, Quick Mart, Carrefour, etc.) with per-item retail prices could be found publicly. If the final application needs branded, current shelf-price data, that would likely have to come from a custom scrape or partnership later in the project — the WFP dataset is the strongest *open* stand-in for now, since it already records real market prices collected in Nairobi.

## 2. Fetching the dataset

The CSV is hosted directly on HDX. Note that WFP/HDX food-price files include an extra **HXL tag row** right after the header (e.g. `#date`, `#adm1+name`) — we skip that row when loading.

*If this direct link ever changes, get the latest one from the "Kenya - Food Prices" page on [data.humdata.org](https://data.humdata.org/dataset/wfp-food-prices-for-kenya) and swap it into `URL` below.*

In [ ]:
import pandas as pd

URL = "https://data.humdata.org/dataset/e0d3fba6-f9a2-45d7-b949-140c455197ff/resource/517ee1bf-2437-4f8c-aa1b-cb9925b9d437/download/wfp_food_prices_ken.csv"

# row 0 = column headers, row 1 = HXL hashtags (not real data) -> skip it
df_all = pd.read_csv(URL, skiprows=[1])
print("Columns available:", list(df_all.columns))
df_all.head()

## 3. Filtering to Nairobi

The full dataset covers markets across Kenya. Our group's scope is Nairobi specifically, so we filter down to rows where `admin1` (county/region) or `market` mentions Nairobi.

In [ ]:
nairobi_mask = (
    df_all["admin1"].astype(str).str.contains("Nairobi", case=False, na=False)
    | df_all["market"].astype(str).str.contains("Nairobi", case=False, na=False)
)

df = df_all[nairobi_mask].reset_index(drop=True)
print(f"Nairobi rows: {len(df)} out of {len(df_all)} total rows")
df.head()

## 4. Data Exploration

### a) How many rows and columns are contained in the dataset?

In [ ]:
num_rows, num_cols = df.shape
print(f"Rows: {num_rows}")
print(f"Columns: {num_cols}")
df.shape

### b) What datatypes are contained in the dataset?

In [ ]:
df.dtypes

### c) Is the dataset complete? i.e. no missing values?

In [ ]:
missing = df.isnull().sum()
print(missing)
print()
is_complete = missing.sum() == 0
print(f"Dataset is complete (no missing values): {is_complete}")

if not is_complete:
    print()
    print("Columns with missing values:")
    print(missing[missing > 0])

### d) Slice out the first 15 rows and last 20 rows, merge into `df_sample`, and display it

In [ ]:
df_sample = pd.concat([df.head(15), df.tail(20)], ignore_index=True)
df_sample